In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Test mode - use subset of data for quick testing
TEST_MODE = True
TEST_CUSTOMERS = 100 if TEST_MODE else None

# Paths
DATA_DIR = '/project/data'
MODEL_DIR = '/project/models'
IMAGE_DIR = f'{DATA_DIR}/collection_images'

# Model parameters
EMBEDDING_DIM = 1280  # MobileNetV2 output
IMG_SIZE = 224
BATCH_SIZE = 32
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'

# Recommendation parameters
TOP_K = 10  # Number of recommendations per customer
MIN_INTERACTIONS = 3  # Minimum interactions to build profile

# Hybrid scoring weights
WEIGHTS = {
    'visual_style': 0.50,       # Image similarity from interaction history
    'collaborative': 0.30,      # Similar customers by behavior
    'behavioral': 0.20,         # Device, traffic source, interaction patterns
}

# Interaction weights (for style profile)
INTERACTION_WEIGHTS = {
    'purchase': 10.0,
    'add_to_cart': 3.0,
    'add_to_wishlist': 5.0,
    'view': 1.0,
    'click': 0.5
}

# Temporal decay
RECENCY_HALF_LIFE_DAYS = 30

# Cold-start thresholds
COLD_START_INTERACTION_THRESHOLD = 5
WARM_START_INTERACTION_THRESHOLD = 20

print(f"⚙️  Configuration loaded")
print(f"  Test Mode: {TEST_MODE}")
print(f"  Device: {DEVICE}")
print(f"  Top-K: {TOP_K}")
print(f"  Hybrid weights: {WEIGHTS}")

⚙️  Configuration loaded
  Test Mode: True
  Device: cpu
  Top-K: 10
  Hybrid weights: {'visual_style': 0.4, 'collaborative': 0.3, 'behavioral': 0.2, 'demographic': 0.1}


In [2]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import json
from datetime import datetime, timedelta
from collections import Counter, defaultdict
from tqdm import tqdm

# PyTorch and vision
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# Scikit-learn
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.neighbors import NearestNeighbors

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Custom modules
import sys
sys.path.append('/project/code')
from customer_style_profiler import CustomerStyleProfiler

# Settings
pd.set_option('display.max_columns', None)
np.random.seed(42)
torch.manual_seed(42)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [ ]:
# ============================================================
# LOAD ALL DATA SOURCES
# ============================================================

print("📂 Loading data sources...")

# 1. Interactions (primary data source from GA4)
interactions = pd.read_csv(f'{DATA_DIR}/sample_interactions.csv')
interactions['timestamp'] = pd.to_datetime(interactions['timestamp'])
interactions = interactions.dropna(subset=['product_id'])  # Remove rows with missing product_id
interactions['product_id'] = interactions['product_id'].astype(int).astype(str)  # Ensure string type

# 2. Product catalog
products = pd.read_csv(f'{DATA_DIR}/feed_a62656-2_de.csv', delimiter=';')

# 3. Build customer profiles from interaction data
print("\n📊 Building customer profiles from interactions...")

# Aggregate customer-level statistics from interactions
customer_stats = []
for customer_id in interactions['customer_id'].unique():
    cust_interactions = interactions[interactions['customer_id'] == customer_id]
    
    # Calculate statistics
    total_interactions = len(cust_interactions)
    purchases = cust_interactions[cust_interactions['interaction_type'] == 'purchase']
    views = cust_interactions[cust_interactions['interaction_type'] == 'view']
    carts = cust_interactions[cust_interactions['interaction_type'] == 'add_to_cart']
    
    # Get product preferences from interactions
    interacted_products = cust_interactions['product_id'].unique()
    
    customer_stats.append({
        'customer_id': customer_id,
        'total_interactions': total_interactions,
        'total_purchases': len(purchases),
        'total_views': len(views),
        'total_carts': len(carts),
        'unique_products': len(interacted_products),
        'first_interaction': cust_interactions['timestamp'].min(),
        'last_interaction': cust_interactions['timestamp'].max(),
        'days_active': (cust_interactions['timestamp'].max() - cust_interactions['timestamp'].min()).days,
        'primary_device': cust_interactions['device_type'].mode()[0] if len(cust_interactions) > 0 else 'mobile',
        'primary_source': cust_interactions['referrer_source'].mode()[0] if len(cust_interactions) > 0 else 'organic'
    })

customers = pd.DataFrame(customer_stats)

# Add engagement metrics for cold-start detection
customers['has_purchases'] = customers['total_purchases'] > 0
customers['is_cold_start'] = customers['total_interactions'] < COLD_START_INTERACTION_THRESHOLD

# Filter by TEST_MODE
if TEST_MODE and TEST_CUSTOMERS:
    print(f"\n⚠️  TEST MODE: Using {TEST_CUSTOMERS} customers")
    
    # Sample customers
    sampled_customer_ids = customers.sample(n=min(TEST_CUSTOMERS, len(customers)), random_state=42)['customer_id'].tolist()
    
    # Filter datasets
    customers = customers[customers['customer_id'].isin(sampled_customer_ids)]
    interactions = interactions[interactions['customer_id'].isin(sampled_customer_ids)]

print(f"\n✓ Data loaded successfully:")
print(f"  Customers: {len(customers):,}")
print(f"  Interactions: {len(interactions):,}")
print(f"  Products in catalog: {len(products):,}")
print(f"  Unique products in interactions: {interactions['product_id'].nunique():,}")
print(f"  Date range: {interactions['timestamp'].min()} to {interactions['timestamp'].max()}")
print(f"\n  Customer Segments:")
print(f"    VIP: {len(customers[customers['customer_segment']=='vip']):,} ({len(customers[customers['customer_segment']=='vip'])/len(customers)*100:.1f}%)")
print(f"\n  Customer Engagement:")
print(f"    With purchases: {len(customers[customers['has_purchases']]):,} ({len(customers[customers['has_purchases']])/len(customers)*100:.1f}%)")print(f"    Cold-start (<{COLD_START_INTERACTION_THRESHOLD} interactions): {len(customers[customers['is_cold_start']]):,} ({len(customers[customers['is_cold_start']])/len(customers)*100:.1f}%)")print(f"    Avg interactions per customer: {customers['total_interactions'].mean():.1f}")

In [ ]:
# ============================================================
# IMAGE EMBEDDING MODEL
# ============================================================

class ImageEmbeddingExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.features = nn.Sequential(*list(mobilenet.children())[:-1])
        self.pool = nn.AdaptiveAvgPool2d(1)
        
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return x

# Initialize model
device = torch.device(DEVICE)
embedding_model = ImageEmbeddingExtractor().to(device)
embedding_model.eval()

print(f"✓ Image embedding model loaded on {DEVICE}")

In [ ]:
# ============================================================
# EXTRACT EMBEDDINGS FOR ALL PRODUCTS
# ============================================================

print("🖼️  Extracting image embeddings...")

# Image preprocessing
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def extract_embedding(image_path):
    """Extract embedding for a single image"""
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = embedding_model(img_tensor)
        return embedding.cpu().numpy().flatten()
    except Exception as e:
        return None

# Get unique products from interactions
unique_products = interactions['product_id'].unique()
image_dir = Path(IMAGE_DIR)

# Extract embeddings
product_embeddings = {}
products_without_images = []

for product_id in tqdm(unique_products, desc="Processing products"):
    image_path = image_dir / f"{product_id}.jpg"
    
    if image_path.exists():
        embedding = extract_embedding(str(image_path))
        if embedding is not None:
            product_embeddings[str(product_id)] = embedding
    else:
        products_without_images.append(product_id)

# Save embeddings
embeddings_file = Path(MODEL_DIR) / "hybrid_product_embeddings.npz"
np.savez_compressed(
    embeddings_file,
    product_ids=list(product_embeddings.keys()),
    embeddings=np.array(list(product_embeddings.values()))
)

print(f"\n✓ Embedding extraction complete:")
print(f"  Products with embeddings: {len(product_embeddings):,}")
print(f"  Products without images: {len(products_without_images):,}")
print(f"  Embedding dimension: {list(product_embeddings.values())[0].shape[0]}")
print(f"  Saved to: {embeddings_file}")

In [ ]:
# ============================================================
# BUILD CUSTOMER PROFILES FROM ALL SIGNALS
# ============================================================

print("👤 Building comprehensive customer profiles...")

# ==================== 1. VISUAL STYLE PROFILE ====================
print("\n1️⃣ Building visual style profiles...")

profiler = CustomerStyleProfiler(
    product_embeddings=product_embeddings,
    recency_half_life_days=RECENCY_HALF_LIFE_DAYS
)
profiler.INTERACTION_WEIGHTS = INTERACTION_WEIGHTS

visual_profiles = profiler.build_all_profiles(
    interactions_df=interactions,
    min_interactions=MIN_INTERACTIONS
)

print(f"  ✓ Visual profiles: {len(visual_profiles):,} customers")

# ==================== 2. BEHAVIORAL PROFILE ====================
print("\n2️⃣ Computing behavioral patterns from interactions...")

behavioral_profiles = {}

for customer_id in customers['customer_id'].unique():
    cust_interactions = interactions[interactions['customer_id'] == customer_id]
    
    # Device preferences
    device_dist = cust_interactions['device_type'].value_counts(normalize=True).to_dict()
    
    # Interaction type distribution
    interaction_dist = cust_interactions['interaction_type'].value_counts(normalize=True).to_dict()
    
    # Product categories (from catalog)
    cust_products = cust_interactions['product_id'].unique()
    product_info = products[products['artikel_id'].astype(str).isin(cust_products)]
    
    # Preferred categories and collections
    preferred_categories = []
    preferred_collections = []
    if len(product_info) > 0 and 'kollektion_de' in product_info.columns:
        preferred_collections = product_info['kollektion_de'].value_counts().head(3).index.tolist()
    
    behavioral_profiles[customer_id] = {
        'device_preferences': device_dist,
        'interaction_patterns': interaction_dist,
        'preferred_collections': preferred_collections,
        'engagement_score': len(cust_interactions) / customers[customers['customer_id']==customer_id]['days_active'].values[0] if customers[customers['customer_id']==customer_id]['days_active'].values[0] > 0 else 0
    }

print(f"  ✓ Behavioral profiles: {len(behavioral_profiles):,} customers")

# ==================== 3. CONTEXT FEATURES (available at inference) ====================
print("\n3️⃣ Extracting context features...")

# These are the only features available at inference time
context_features = customers[['customer_id', 'total_interactions', 'total_purchases',
                               'days_active', 'primary_device', 'primary_source']].copy()

print(f"  ✓ Context features: {len(context_features):,} customers")

print(f"\n✅ Profile building complete!")
print(f"  Customers with visual profiles: {len(visual_profiles):,}")
print(f"  Customers with behavioral profiles: {len(behavioral_profiles):,}")
print(f"  Customers with context features: {len(context_features):,}")
print(f"\n  Cold-start customers (need fallback): {len(customers[customers['is_cold_start']]):,}")

## 5️⃣ Build Multi-Signal Customer Profiles

## 4️⃣ Extract Image Embeddings

## 3️⃣ Load All Data Sources

## 2️⃣ Import Libraries

## 1️⃣ Configuration

# 🎯 Hybrid Recommendation System
## Multi-Signal Product Recommendations for E-commerce

This notebook implements a **production-ready hybrid recommendation engine** that combines:

### 📊 Data Sources
- **Visual Style**: Image embeddings (MobileNetV2) for visual similarity
- **Behavioral Data**: Customer interactions (views, carts, purchases)
- **Demographics**: Age, gender, location, income, customer segment
- **Purchase History**: Orders, AOV, preferred categories, materials
- **Product Catalog**: Full jewelry collection with metadata

### 🎨 Recommendation Strategies

**1️⃣ Visual Style Matching (40%)**
- MobileNetV2 embeddings from product images
- Weighted by interaction type (purchase > cart > view)
- Recency decay for temporal relevance

**2️⃣ Collaborative Filtering (30%)**
- Find similar customers by purchase/interaction patterns
- "Customers like you also bought..."
- Matrix factorization on interaction data

**3️⃣ Behavioral Patterns (20%)**
- Preferred categories, materials, price ranges
- Brand affinity and collection preferences
- Purchase frequency and seasonality

**4️⃣ Demographic Matching (10%)**
- Age group, gender, location
- Customer segment (first-time, regular, VIP)
- Income bracket and spending patterns

### 🚀 Key Features
- **Cold-start handling** for new customers
- **Explainable recommendations** with scoring breakdown
- **A/B testing ready** with performance metrics
- **Real-time scoring** optimized for production

---

**Goal**: Maximize conversion rate and customer satisfaction by showing the most relevant products to each customer.